# World Cup 2026 scraping

## Setup

In [1]:
import requests
import json

from itertools import dropwhile, takewhile
import re

from typing import Callable, Iterable, Dict, List, Any

In [9]:
def is_landmark(
    value: str, nodetype: str = "heading-2"
) -> Callable[[dict], bool]:
    def f(x: dict) -> bool:
        res = not (
            x["nodeType"] == nodetype and
            x["content"][0]["value"] == value
        )
        return res
    return f

In [2]:
s = requests.Session()

headers = {
    "Accept": 'application/json, text/plain, */*',
    "Accept-Encoding": 'gzip, deflate, br, zstd',
    "Accept-Language": 'pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7,es-ES;q=0.6,es;q=0.5',
    "Connection":      'keep-alive',
    "DNT": '1',
    "Host":   'cxm-api.fifa.com',
    "Origin": 'https://www.fifa.com',
    "Sec-Fetch-Dest": 'empty',
    "Sec-Fetch-Mode": 'cors',
    "Sec-Fetch-Site": 'same-site',
    "User-Agent": 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36',
    "sec-ch-ua": '"Google Chrome";v="149", "Chromium";v="149", "Not)A;Brand";v="24"',
    "sec-ch-ua-mobile": '?0',
    "sec-ch-ua-platform": '"Windows"'
}

s = requests.Session()
s.headers.update(headers)

## Group Stage

In [4]:
url = "https://cxm-api.fifa.com/fifaplusweb/api/sections/article/S9YG2JmeGYaMUCBbm0CcD?locale=pt"

request_get = s.get(url)

if request_get.status_code != 200:
    raise Exception("Failed to fetch countries")

content = json.loads(request_get.content)

In [ ]:
content_content: list = content["richtext"]["content"]

content_filtered: Iterable = dropwhile(is_landmark("Fase de grupos"), content_content)
next(content_filtered)
content_filtered = takewhile(is_landmark("Início do mata-mata"), content_filtered)

In [12]:
data_days: Dict[str, list] = {}
current_day: str | None = None

for x in content_filtered:
    if x["nodeType"] == "heading-4":
        current_day = x["content"][0]["value"]
        if not isinstance(current_day, str):
            raise Exception("Expected heading to be a string")
        data_days[current_day] = []
    else:
        if x["nodeType"] == "paragraph" and x["content"][0]["value"] != "":
            if not isinstance(current_day, str):
                raise Exception("Paragraph found before heading 4")
            data_days[current_day].append(x)

In [13]:
data_fmt: Dict[str, List[Dict[str, Any]]] = {}

for dk, dv in data_days.items():
    data_fmt[dk] = []

    for xi, xv in enumerate(dv):
        data = {}

        xc = [
            x
            for x in xv["content"]
            if (
                (x["nodeType"] == "hyperlink" or re.search(r" x ", x["value"])) or
                re.search(r"^Grupo ([A-Z]): $", x["value"]) or
                re.search(r"([0-9]{1,2}h[0-9]{2})[^/]+Brasília", x["value"])
            )
        ]
        if (len(xc) != 3):
            raise Exception(
                f"Error at day {dk} and index {xi}:\n"
                f"x - Expected 3 content items, got {len(xc)}\n"
                f"i - Content: {xc}"
            )

        group = re.search(r"^Grupo ([A-Z]): $", xc[0]["value"])
        if not group:
            raise Exception(
                f"Error at day {dk} and index {xi}:\n"
                f"x - No group match\n"
                f"i - Content: {xc[0]}"
            )
        data["group"] = group.group(0)

        if xc[1]["nodeType"] == "hyperlink":
            teams = xc[1]["content"][0]["value"]
        else:
            teams = xc[1]["value"]
        if not re.search(r" x ", teams):
            raise Exception(
                f"Error at day `{dk}` and index `{xi}`:\n"
                f"x - No teams match\n"
                f"i - Content: {xc[1]}"
            )
        data["teams"] = teams
        
        time = re.search(r"([0-9]{1,2}h[0-9]{2})[^/]+Brasília", xc[2]["value"])
        if not time:
            raise Exception(
                f"Error at day {dk} and index {xi}:\n"
                f"x - No time match\n"
                f"i - Content: {xc[2]}"
            )
        data["time"] = time.group(0)

        result = re.search(r" ([0-9]+) x ([0-9]+) ", teams)
        data["result"] = result.groups() if result else None
        
        data_fmt[dk].append(data)

In [14]:
teams = r"Equador|Brasil|Japão|Congo|Escócia|Cabo Verde"

{
    k: [
        tuple(m.values())
        for m in x
        if (
            re.search(teams, m["teams"]) and
            m["result"] is None
        )
    ]
    for k, x in data_fmt.items()
}

{}

## Knockout stage

In [7]:
url = "https://cxm-api.fifa.com/fifaplusweb/api/sections/article/S9YG2JmeGYaMUCBbm0CcD?locale=en"

request_get = s.get(url)

if request_get.status_code != 200:
    raise Exception("Failed to fetch countries")

content = json.loads(request_get.content)

In [39]:
content_content: list = content["richtext"]["content"]

content_filtered: Iterable = dropwhile(
    is_landmark("FIFA World Cup 2026 – Round of 32 fixtures", "heading-3"),
    content_content
)
#content_filtered = takewhile(is_landmark("Início do mata-mata"), content_filtered)

In [41]:
data_days: Dict[str, list] = {}
current_day: str | None = None

for x in content_filtered:
    if x["nodeType"] == "heading-4":
        current_day = x["content"][0]["value"]
        if not isinstance(current_day, str):
            raise Exception("Expected heading to be a string")
        data_days[current_day] = []
    else:
        if x["nodeType"] == "paragraph" and x["content"][0]["value"] != "":
            if not isinstance(current_day, str):
                raise Exception("Paragraph found before heading 4")
            data_days[current_day].append(x)

In [34]:
data_days

{'Sunday, 28 June 2026': [{'nodeType': 'paragraph',
   'data': {},
   'content': [{'nodeType': 'text',
     'data': {},
     'value': 'Match 73 – ',
     'marks': []},
    {'nodeType': 'hyperlink',
     'data': {'uri': 'https://www.fifa.com/en/match-centre/match/17/285023/289287/400021518'},
     'content': [{'nodeType': 'text',
       'data': {},
       'value': 'South Africa 0-1 Canada',
       'marks': []}]},
    {'nodeType': 'text',
     'data': {},
     'value': ' - Los Angeles Stadium',
     'marks': []}]}],
 'Monday, 29 June 2026': [{'nodeType': 'paragraph',
   'data': {},
   'content': [{'nodeType': 'text',
     'data': {},
     'value': 'Match 74 – ',
     'marks': []},
    {'nodeType': 'hyperlink',
     'data': {'uri': 'https://www.fifa.com/en/match-centre/match/17/285023/289287/400021513'},
     'content': [{'nodeType': 'text',
       'data': {},
       'value': 'Germany v Paraguay',
       'marks': []}]},
    {'nodeType': 'text',
     'data': {},
     'value': ' - 16:30 - B

In [35]:
data_fmt: Dict[str, List[Dict[str, Any]]] = {}

for dk, dv in data_days.items():
    data_fmt[dk] = []

    for xi, xv in enumerate(dv):
        data = {}

        xc = [
            x
            for x in xv["content"]
            if (
                (x["nodeType"] == "hyperlink" or re.search(r" x ", x["value"])) or
                re.search(r"^Grupo ([A-Z]): $", x["value"]) or
                re.search(r"([0-9]{1,2}h[0-9]{2})[^/]+Brasília", x["value"])
            )
        ]
        if (len(xc) != 3):
            raise Exception(
                f"Error at day {dk} and index {xi}:\n"
                f"x - Expected 3 content items, got {len(xc)}\n"
                f"i - Content: {xc}"
            )

        group = re.search(r"^Grupo ([A-Z]): $", xc[0]["value"])
        if not group:
            raise Exception(
                f"Error at day {dk} and index {xi}:\n"
                f"x - No group match\n"
                f"i - Content: {xc[0]}"
            )
        data["group"] = group.group(0)

        if xc[1]["nodeType"] == "hyperlink":
            teams = xc[1]["content"][0]["value"]
        else:
            teams = xc[1]["value"]
        if not re.search(r" x ", teams):
            raise Exception(
                f"Error at day `{dk}` and index `{xi}`:\n"
                f"x - No teams match\n"
                f"i - Content: {xc[1]}"
            )
        data["teams"] = teams
        
        time = re.search(r"([0-9]{1,2}h[0-9]{2})[^/]+Brasília", xc[2]["value"])
        if not time:
            raise Exception(
                f"Error at day {dk} and index {xi}:\n"
                f"x - No time match\n"
                f"i - Content: {xc[2]}"
            )
        data["time"] = time.group(0)

        result = re.search(r" ([0-9]+) x ([0-9]+) ", teams)
        data["result"] = result.groups() if result else None
        
        data_fmt[dk].append(data)

Exception: Error at day Sunday, 28 June 2026 and index 0:
x - Expected 3 content items, got 1
i - Content: [{'nodeType': 'hyperlink', 'data': {'uri': 'https://www.fifa.com/en/match-centre/match/17/285023/289287/400021518'}, 'content': [{'nodeType': 'text', 'data': {}, 'value': 'South Africa 0-1 Canada', 'marks': []}]}]

In [29]:
teams = r"Equador|Brasil|Japão|Congo|Escócia|Cabo Verde"

{
    k: [
        tuple(m.values())
        for m in x
        if (
            re.search(teams, m["teams"]) and
            m["result"] is None
        )
    ]
    for k, x in data_fmt.items()
}

{}